# Estimating Obesity Levels from Eating Habits and Physical Condition

**Goal:** classify people into 7 weight categories (`NObeyesdad`) using the UCI *Estimation of obesity levels* dataset.

**Two questions are answered in this notebook**
1. How well can we classify weight category using *all* features (including Height and Weight)?
2. How well can we classify it using **eating habits and lifestyle only** (Height and Weight removed)?

> Note: the weight categories are defined by BMI cutoffs (`Weight / Height²`), so Question 1 is partly circular. Question 2 is the more informative one. About 77% of this dataset was also synthetically generated (SMOTE) by its original authors, so scores may be somewhat optimistic.

## 1. Setup and data loading

In [ ]:
# Importing all libraries once
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, confusion_matrix, cohen_kappa_score,
                             f1_score, accuracy_score, make_scorer)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sns.set_theme(style="whitegrid")

In [ ]:
# Loading the data.
# Put the CSV in the same folder as this notebook, or edit the fallback path below.
local_name = "ObesityDataSet_raw_and_data_sinthetic.csv"
fallback_path = r"C:\Users\Home\Downloads\estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition\ObesityDataSet_raw_and_data_sinthetic.csv"

file_path = local_name if os.path.exists(local_name) else fallback_path
obesity = pd.read_csv(file_path)

## 2. Exploratory data analysis

In [ ]:
# Printing the shape of the data and displaying it's head
print("Shape of the data:", obesity.shape)
display(obesity.head())
obesity.info()

In [ ]:
# Checking the dataset for missing values, duplicate rows, and target class distribution
print("=== DATA QUALITY CHECKS ===")
print("Missing values per column:\n", obesity.isnull().sum())
print(f"\nDuplicate Rows Count: {obesity.duplicated().sum()}")

print("\n=== TARGET CLASS DISTRIBUTION (NObeyesdad) ===")
print(obesity['NObeyesdad'].value_counts())
print("\nTarget Class Proportions (%):\n", (obesity['NObeyesdad'].value_counts(normalize=True) * 100).round(2))

In [ ]:
# Calculating standard BMI to confirm consistency in data
print("=== SUMMARY OF NUMERICAL FEATURES ===")
display(obesity.describe().T)

obesity['Calculated_BMI'] = obesity['Weight'] / (obesity['Height'] ** 2)

print("\n=== CALCULATED BMI SUMMARY BY OBESITY CATEGORY ===")
display(obesity.groupby('NObeyesdad')['Calculated_BMI'].agg(['min', 'mean', 'max', 'std']).round(2))

The BMI ranges per class are almost perfectly ordered, which confirms the target is essentially a BMI cutoff. Any model given Height and Weight can therefore recover most of the label directly from them. That is why a second, lifestyle-only model is trained later in this notebook.

In [ ]:
# Evaluating outliers using the IQR method
print("=== Evaluating for Outliers (IQR method) ===")

num_cols = obesity.select_dtypes(include=[np.number]).columns.drop('Calculated_BMI', errors='ignore')

outlier_sum = []

for col in num_cols:
    Q1 = obesity[col].quantile(0.25)
    Q3 = obesity[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier = obesity[(obesity[col] < lower_bound) | (obesity[col] > upper_bound)]
    outlier_pct = (len(outlier) / len(obesity)) * 100

    outlier_sum.append({
        'Feature': col,
        'Q1 (25%)': round(Q1, 2),
        'Q3 (75%)': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Lower Bound': round(lower_bound, 2),
        'Upper Bound': round(upper_bound, 2),
        'Count of Outliers': len(outlier),          # FIX: was round(outlier), which rounded the whole DataFrame
        'Outlier %': round(outlier_pct, 2),
    })

outlier_df = pd.DataFrame(outlier_sum)
display(outlier_df)

Outliers are **kept**: they are real, plausible values (for example older participants or very high weights) and tree-based models are robust to them.

In [ ]:
# Bar plots for categorical features
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

cat_cols = obesity.select_dtypes(include=['object', 'category', 'str']).columns.drop('NObeyesdad', errors='ignore')
for col in cat_cols:
    plt.figure(figsize=(10, 5))
    sns.countplot(data=obesity, x=col, hue='NObeyesdad', palette="viridis")
    plt.title(f'Distribution of {col} across Obesity Categories')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='NObeyesdad', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
# Box plots for continuous features
num_cols = obesity.select_dtypes(include=['number']).columns.drop('Calculated_BMI', errors='ignore')

for num in num_cols:
    plt.figure(figsize=(12, 6))
    # FIX: previously y=col (a leftover variable), so every plot showed the same feature
    sns.boxplot(data=obesity, x='NObeyesdad', y=num, hue='NObeyesdad', palette="Set3", legend=False)
    plt.title(f'Box Plot of {num} by Obesity Weight Category')
    plt.xlabel('Obesity Classification (NObeyesdad)')
    plt.ylabel(num)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap of the numeric features (and BMI)
plt.figure(figsize=(8, 6))
sns.heatmap(obesity.select_dtypes(include=[np.number]).corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.show()

## 3. Data preparation

- Duplicate rows are dropped before splitting.
- The target is mapped to ordered integers (0 = Insufficient_Weight … 6 = Obesity_Type_III).
- `CAEC` and `CALC` are **ordinal** (no < Sometimes < Frequently < Always).
- Yes/no columns and `Gender` are mapped to 0/1; `MTRANS` is **nominal**, so it is one-hot encoded.
- Min-max scaling and encoding happen inside the pipeline, fitted on training data only.

In [ ]:
# Data cleaning and encoding
df = obesity.drop_duplicates().copy()
df = df.drop(columns=['Calculated_BMI'], errors='ignore')

target_map = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6
}
df['NObeyesdad'] = df['NObeyesdad'].map(target_map)

freq_map = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
df['CAEC'] = df['CAEC'].map(freq_map)
df['CALC'] = df['CALC'].map(freq_map)

binary_col = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
for col in binary_col:
    df[col] = df[col].map({'no': 0, 'yes': 1})

df['Gender'] = df['Gender'].map({'Female': 0, 'Male': 1})

print("Completed Target and Ordinal Encoding. Checking for missing values:", df.isnull().sum().sum())

In [ ]:
# Splitting the dataset using a stratified train-test split
X = df.drop(columns=['NObeyesdad'])
y = df['NObeyesdad']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Shape of Training Set: {X_train.shape}")
print(f"Shape of Testing Set: {X_test.shape}")

In [ ]:
# Scaling continuous numerical features to the [0,1] range and applying One-Hot Encoding to nonimal categorical variables
numer_cols = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
nominal_cols = ['MTRANS']
passingthrough_cols = ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numer_cols),
        ('nom', OneHotEncoder(drop='first', sparse_output=False), nominal_cols),
        ('pass', 'passthrough', passingthrough_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

encode_mtrans_names = preprocessor.named_transformers_['nom'].get_feature_names_out(nominal_cols)
all_feat_names = numer_cols + list(encode_mtrans_names) + passingthrough_cols

X_train_df = pd.DataFrame(X_train_processed, columns=all_feat_names)
X_test_df = pd.DataFrame(X_test_processed, columns=all_feat_names)

print("Shape of Preprocessed Training Features:", X_train_df.shape)
display(X_train_df.head())

In [ ]:
# Applying SMOTE on the training set
smote = SMOTE(random_state=42)
X_train_balance, y_train_balance = smote.fit_resample(X_train_df, y_train)

print("=== Class Counts Before Applying SMOTE (Training Set) ===")
print(y_train.value_counts().sort_index())

print("\n=== Class Counts After Applying SMOTE (Training Set) ===")
print(pd.Series(y_train_balance).value_counts().sort_index())

The SMOTE cell above only *demonstrates* the class balancing. During cross-validation SMOTE is applied inside the pipeline, to training folds only, so no synthetic rows leak into validation data.

## 4. Model A: all features (including Height and Weight)

In [ ]:
# Leak-free cross-validation of three candidate models
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42)   # default hyperparameters, not tuned
}

crossVal = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'f1_macro': 'f1_macro',
    'cohen_kappa': make_scorer(cohen_kappa_score)
}

crossVal_res = []

for name, clf in models.items():
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', clf)
    ])

    scores = cross_validate(pipeline, X_train, y_train, cv=crossVal, scoring=scoring, n_jobs=-1)

    crossVal_res.append({
        'Model': name,
        'Mean Accuracy': round(scores['test_accuracy'].mean(), 4),
        'Std Accuracy': round(scores['test_accuracy'].std(), 4),
        'Mean Macro F1': round(scores['test_f1_macro'].mean(), 4),
        'Mean Cohen Kappa': round(scores['test_cohen_kappa'].mean(), 4)
    })

crossVal_df = pd.DataFrame(crossVal_res).sort_values(by='Mean Macro F1', ascending=False).reset_index(drop=True)
display(crossVal_df)

In [ ]:
# Tuning hyperparameters for top model

from sklearn.model_selection import GridSearchCV

rf_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 15, 25],
    'classifier__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=crossVal,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Best Cross-Validated Macro F1:", round(grid_search.best_score_, 4))
print("Best Hyperparameters:", grid_search.best_params_)

In [ ]:
# Evaluating the tuned pipeline on the hold-out test set
# (GridSearchCV has already refit the best pipeline on the full training set)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

target_names = [
    'Insufficient_Weight', 'Normal_Weight', 'Overweight_Level_I',
    'Overweight_Level_II', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III'
]

print("=== MODEL A (ALL FEATURES): HOLD-OUT TEST METRICS ===")
print(f"Test Accuracy:      {accuracy_score(y_test, y_pred):.4f}")
print(f"Test Macro F1:      {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"Test Cohen's Kappa: {cohen_kappa_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=target_names, digits=4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Model A (All Features): Confusion Matrix on Hold-Out Test Set')
plt.xlabel('Predicted Category')
plt.ylabel('Actual Category')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for Model A: Gini importance (from the forest) ...
rf_clf = best_model.named_steps['classifier']
prep = best_model.named_steps['preprocessor']

encoded_nom = prep.named_transformers_['nom'].get_feature_names_out(nominal_cols)
feature_names = numer_cols + list(encoded_nom) + passingthrough_cols

importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_clf.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importances.head(12), x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)
plt.title('Model A: Top 12 Features (Gini Importance)')
plt.xlabel('Gini Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# ... and permutation importance on the hold-out test set (less biased than Gini)
perm_A = permutation_importance(best_model, X_test, y_test, scoring='f1_macro',
                                n_repeats=10, random_state=42, n_jobs=-1)
perm_A_df = (pd.DataFrame({'Feature': X_test.columns, 'Importance': perm_A.importances_mean})
             .sort_values('Importance', ascending=False))

plt.figure(figsize=(10, 6))
sns.barplot(data=perm_A_df.head(12), x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)
plt.title('Model A: Permutation Importance (drop in test macro F1)')
plt.xlabel('Mean decrease in macro F1')
plt.tight_layout()
plt.show()

## 5. Model B: eating habits and lifestyle only (no Height, no Weight)

Because the labels are BMI cutoffs, Model A is partly just rediscovering the BMI formula. Here Height and Weight are **removed** so the model must rely only on age, gender, family history, eating habits, physical activity, technology use, and transport.

Everything else is kept identical to Model A (same train/test split, same CV folds, same SMOTE-in-pipeline setup, same candidate models and tuning grid) so the two are directly comparable.

In [ ]:
# Building the lifestyle-only feature set and preprocessor
drop_cols = ['Height', 'Weight']

X_train_B = X_train.drop(columns=drop_cols)
X_test_B = X_test.drop(columns=drop_cols)

numer_cols_B = [c for c in numer_cols if c not in drop_cols]      # Age, FCVC, NCP, CH2O, FAF, TUE
# nominal_cols and passingthrough_cols stay the same

preprocessor_B = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numer_cols_B),
        ('nom', OneHotEncoder(drop='first', sparse_output=False), nominal_cols),
        ('pass', 'passthrough', passingthrough_cols)
    ]
)

print("Model B features:", list(X_train_B.columns))
print("Training shape:", X_train_B.shape, "| Test shape:", X_test_B.shape)

In [ ]:
# Cross-validating the same three candidate models without Height and Weight
crossVal_res_B = []

for name, clf in models.items():
    pipeline_B = ImbPipeline(steps=[
        ('preprocessor', preprocessor_B),
        ('smote', SMOTE(random_state=42)),
        ('classifier', clf)
    ])

    scores = cross_validate(pipeline_B, X_train_B, y_train, cv=crossVal, scoring=scoring, n_jobs=-1)

    crossVal_res_B.append({
        'Model': name,
        'Mean Accuracy': round(scores['test_accuracy'].mean(), 4),
        'Std Accuracy': round(scores['test_accuracy'].std(), 4),
        'Mean Macro F1': round(scores['test_f1_macro'].mean(), 4),
        'Mean Cohen Kappa': round(scores['test_cohen_kappa'].mean(), 4)
    })

crossVal_df_B = pd.DataFrame(crossVal_res_B).sort_values(by='Mean Macro F1', ascending=False).reset_index(drop=True)
display(crossVal_df_B)

In [ ]:
# Tuning a Random Forest on the lifestyle-only features (same grid as Model A)
rf_pipeline_B = ImbPipeline(steps=[
    ('preprocessor', preprocessor_B),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

grid_search_B = GridSearchCV(
    estimator=rf_pipeline_B,
    param_grid=param_grid,
    cv=crossVal,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search_B.fit(X_train_B, y_train)
print("Best Cross-Validated Macro F1 (Model B):", round(grid_search_B.best_score_, 4))
print("Best Hyperparameters (Model B):", grid_search_B.best_params_)

In [ ]:
# Evaluating Model B on the same hold-out test set
best_model_B = grid_search_B.best_estimator_
y_pred_B = best_model_B.predict(X_test_B)

print("=== MODEL B (LIFESTYLE ONLY): HOLD-OUT TEST METRICS ===")
print(f"Test Accuracy:      {accuracy_score(y_test, y_pred_B):.4f}")
print(f"Test Macro F1:      {f1_score(y_test, y_pred_B, average='macro'):.4f}")
print(f"Test Cohen's Kappa: {cohen_kappa_score(y_test, y_pred_B):.4f}\n")
print(classification_report(y_test, y_pred_B, target_names=target_names, digits=4))

cm_B = confusion_matrix(y_test, y_pred_B)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_B, annot=True, fmt='d', cmap='Oranges', xticklabels=target_names, yticklabels=target_names)
plt.title('Model B (No Height/Weight): Confusion Matrix on Hold-Out Test Set')
plt.xlabel('Predicted Category')
plt.ylabel('Actual Category')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Which lifestyle factors matter most? (permutation importance on the hold-out test set)
perm_B = permutation_importance(best_model_B, X_test_B, y_test, scoring='f1_macro',
                                n_repeats=10, random_state=42, n_jobs=-1)
perm_B_df = (pd.DataFrame({'Feature': X_test_B.columns, 'Importance': perm_B.importances_mean})
             .sort_values('Importance', ascending=False))

plt.figure(figsize=(10, 6))
sns.barplot(data=perm_B_df, x='Importance', y='Feature', hue='Feature', palette='magma', legend=False)
plt.title('Model B: Permutation Importance (drop in test macro F1)')
plt.xlabel('Mean decrease in macro F1')
plt.tight_layout()
plt.show()

## 6. Comparing Model A and Model B

In [ ]:
# Side-by-side comparison on the same hold-out test set
comparison = pd.DataFrame({
    'Model': ['A: All features', 'B: Lifestyle only (no Height/Weight)'],
    'Test Accuracy': [accuracy_score(y_test, y_pred), accuracy_score(y_test, y_pred_B)],
    'Test Macro F1': [f1_score(y_test, y_pred, average='macro'), f1_score(y_test, y_pred_B, average='macro')],
    'Test Cohen Kappa': [cohen_kappa_score(y_test, y_pred), cohen_kappa_score(y_test, y_pred_B)],
}).round(4)
display(comparison)

# Per-class F1 for both models
per_class = pd.DataFrame({
    'Model A F1': f1_score(y_test, y_pred, average=None),
    'Model B F1': f1_score(y_test, y_pred_B, average=None),
}, index=target_names).round(4)
per_class['Drop (A - B)'] = (per_class['Model A F1'] - per_class['Model B F1']).round(4)
display(per_class)

per_class[['Model A F1', 'Model B F1']].plot(kind='bar', figsize=(10, 5))
plt.title('Per-Class F1: All Features vs. Lifestyle Only')
plt.ylabel('F1 score')
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Conclusions

*(Write your findings here after running the notebook. Suggested points:)*
- How much did accuracy / macro F1 / kappa fall when Height and Weight were removed?
- Which classes suffered most? (Compare the per-class F1 table and the two confusion matrices.)
- Which lifestyle features carry the most signal in Model B (permutation importance chart)?
- Caveats: the labels are BMI-based, ~77% of rows are synthetic, and the test set may contain near-copies of synthetic training rows, so scores are likely optimistic.